In [1]:
# Install dependencies
!pip install streamlit mtcnn keras-facenet joblib pyngrok -q
!npm install -g localtunnel -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.1 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠹
⠹3 packages are looking for funding
⠹  run `npm fund` for details
⠹npm notice
npm notice New major version of npm available! 10.8.2 -> 11.14.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.14.0
npm notice To update run: npm install -g npm@11.14.0
npm notice
⠹

In [2]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# CELL 3: Write the Streamlit app
streamlit_code = '''
import streamlit as st
import cv2
import numpy as np
import joblib
from PIL import Image
from mtcnn import MTCNN
from keras_facenet import FaceNet

st.title("🌟 Celebrity Face Recognizer")
st.write("Upload a photo to see if our AI recognizes the celebrity!")

@st.cache_resource
def load_models():
    detector = MTCNN()
    embedder = FaceNet()
    model_path  = "/content/drive/MyDrive/Face_Recognition_Project/face_recognizer_model.pkl"
    encoder_path = "/content/drive/MyDrive/Face_Recognition_Project/label_encoder.pkl"
    svm_model = joblib.load(model_path)
    encoder   = joblib.load(encoder_path)
    return detector, embedder, svm_model, encoder

detector, embedder, svm_model, encoder = load_models()

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")
    frame = np.array(image)
    st.image(image, caption="Uploaded Image", use_column_width=True)

    with st.spinner("Analyzing face..."):
        try:
            results = detector.detect_faces(frame)
        except Exception as e:
            st.error(f"Face detection error: {e}")
            results = []

        if results:
            for result in results:
                x, y, w, h = result["box"]
                x, y = max(0, x), max(0, y)
                face = frame[y:y+h, x:x+w]

                if face.size == 0 or face.shape[0] < 48 or face.shape[1] < 48:
                    st.warning("Detected face crop is too small — skipping.")
                    continue

                face_resized  = cv2.resize(face, (160, 160))
                face_pixels   = np.expand_dims(face_resized, axis=0)
                embedding     = embedder.embeddings(face_pixels)[0].reshape(1, -1)

                prediction  = svm_model.predict(embedding)
                probability = svm_model.predict_proba(embedding)[0]
                name        = encoder.inverse_transform(prediction)[0]
                confidence  = np.max(probability) * 100

                if confidence > 50:
                    st.success(f"Recognized: **{name}** ({confidence:.2f}%)")
                else:
                    st.warning("Face detected, but confidence is too low to identify.")
        else:
            st.error("No faces found in this image.")
'''

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_code.strip())

print("✅ streamlit_app.py written.")





✅ streamlit_app.py written.


In [3]:
# Start Streamlit and ngrok
from pyngrok import ngrok
import time

In [4]:
# Add the authtoken
ngrok.set_auth_token("3DJyPRKndMNfs2HiRHwJu7w84Sb_8CMEwGAb3vZkXWKuLzPH")

# Kill any old processes/tunnels
!pkill -f streamlit 2>/dev/null
ngrok.kill()
time.sleep(2)

# ✅ START STREAMLIT as a background process on port 8501
!streamlit run streamlit_app.py \
    --server.port 8501 \
    --server.headless true \
    --server.enableCORS false \
    &>/content/streamlit.log &

time.sleep(5)  # Give Streamlit time to boot

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print(f"🔗 Open your app here: {public_url}")

^C
🔗 Open your app here: NgrokTunnel: "https://register-false-colony.ngrok-free.dev" -> "http://localhost:8501"
